# Shor's Algorithm for the Elliptic-Curve Discrete Log — a Toy Demonstration on Aer

This notebook runs the **discrete-logarithm** variant of Shor's algorithm (the one that
attacks **ECDLP**) on the smallest non-trivial elliptic curve that a statevector simulator
can actually handle. It is the companion to the $N=21$ *factoring* notebook: same algorithmic
machinery, different group.

---

## ⚠️ Read this first: why this is a *toy*, not "breaking ECDLP-256"

**ECDLP-256 (e.g. `secp256k1`, the curve behind Bitcoin and much of TLS) cannot be run on Aer,
or on any classical simulator, now or ever.** This is not a matter of patience or a bigger laptop:

| | This toy curve | Real ECDLP-256 |
|---|---|---|
| Field size | $p = 5$ | $p \approx 2^{256}$ |
| Group order $r$ | $7$ | $\approx 2^{256}$ |
| Logical qubits needed | ~12 | **~2,330** (Roetteler et al., 2017) |
| Toffoli gates | a handful | **~$1.3\times10^{11}$** |
| Aer statevector limit | fine | $2^{2330}$ amplitudes — more than the number of atoms in the universe |

A 256-bit run would need a fault-tolerant machine with *millions* of physical qubits running
for hours. Today's hardware is nowhere close, which is exactly why elliptic-curve cryptography
is still secure. **This toy gives zero practical uplift against real ECC** — in the same way
that factoring 21 gives none against RSA-2048. What it *does* show is the genuine mechanism:
the quantum routine recovers a secret scalar $k$ purely from the group structure.

We pick a secret $k$, publish only $P$ and $Q=kP$, and let the quantum algorithm recover $k$.

In [ ]:
# Colab setup
!pip install -q qiskit qiskit-aer pylatexenc

## 1. A tiny elliptic curve and its group

Curve $E:\;y^2 = x^3 + 2x + 1 \pmod 5$. Its points form a **cyclic group of prime order $r=7$**
with generator $P=(0,1)$. We choose secret $k=5$ and publish $Q=kP$. The discrete-log problem:
*given $P$ and $Q$, find $k$.*

In [ ]:
import numpy as np
from collections import Counter
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import UnitaryGate
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

p, A_, B_ = 5, 2, 1            # curve y^2 = x^3 + 2x + 1 over F_5
r = 7                          # prime order of the cyclic group
k_secret = 5                   # the scalar the quantum algorithm must recover

def inv(t, m): return pow(t, -1, m)
def ec_add(P, Q):
    if P is None: return Q
    if Q is None: return P
    x1, y1 = P; x2, y2 = Q
    if x1 == x2 and (y1 + y2) % p == 0: return None      # P + (-P) = O
    if P == Q: lam = (3*x1*x1 + A_) * inv((2*y1) % p, p) % p
    else:      lam = (y2 - y1) * inv((x2 - x1) % p, p) % p
    x3 = (lam*lam - x1 - x2) % p
    y3 = (lam*(x1 - x3) - y1) % p
    return (x3, y3)
def ec_mul(k, P):
    R = None
    for _ in range(k % r): R = ec_add(R, P)
    return R

P = (0, 1)
group = [ec_mul(i, P) for i in range(r)]   # [O, P, 2P, ..., 6P]
Q = ec_mul(k_secret, P)
print("Cyclic group <P> of order", r, ":")
for i, T in enumerate(group):
    print(f"  {i}P = {('O' if T is None else T)}")
print(f"\nPublished:  P = {P},  Q = kP = {Q}")
print(f"Secret (hidden from the algorithm):  k = {k_secret}")

## 2. The algorithm

Shor's discrete-log routine uses two index registers $a,b\in\mathbb{Z}_r$ and a point register:

1. Prepare $\frac{1}{r}\sum_{a,b}|a\rangle|b\rangle|\mathcal{O}\rangle$.
2. Compute $f(a,b)=aP+bQ=(a+bk)P$ into the point register, via **controlled elliptic-curve
   point additions** (add $2^j P$ controlled on bit $j$ of $a$; add $2^j Q$ controlled on bit
   $j$ of $b$).
3. Apply the order-$r$ QFT to both index registers and measure.

A short calculation (the point register depends only on $m=a+bk \bmod r$) shows the measured
pair $(c,d)$ always satisfies
$$d \equiv k\,c \pmod r,$$
so for any $c\neq0$ (and $r$ prime) we recover $k = d\,c^{-1} \bmod r$. The $(0,0)$ outcome
carries no information and occurs with probability $1/r$.

The point register encodes a point $(x,y)$ as the integer $8x+y$ (6 qubits), with the point at
infinity $\mathcal{O}$ reserved as code $63=111111_2$. Each controlled "add $S$" is the exact
elliptic-curve point-addition permutation on the 7 group points — real EC arithmetic, not a
discrete-log shortcut.

In [ ]:
def enc(T): return 63 if T is None else T[0]*8 + T[1]
def dec(c):  return None if c == 63 else (c//8, c%8)
codes = {enc(T) for T in group}

def add_S_perm(S):
    '''64x64 permutation: point register T -> T + S (identity on unused codewords).'''
    M = np.eye(64)
    for c in range(64):
        if c in codes or c == 63:
            U = ec_add(dec(c), S)
            M[:, c] = 0; M[enc(U), c] = 1
    return M

def controlled_add(S):
    '''128x128 controlled-(add S). Control = gate qubit 0 (LSB): index = pt*2 + ctrl.'''
    sub = add_S_perm(S)
    P0 = np.diag([1, 0]); P1 = np.diag([0, 1])
    return np.kron(sub, P1) + np.kron(np.eye(64), P0)

# uniform-over-{0..6} state-prep unitary on 3 qubits (first column = uniform on 0..6)
def prep7():
    M = np.eye(8); M[:, 0] = 0; M[:7, 0] = 1/np.sqrt(7)
    for i in range(1, 8):                       # Gram-Schmidt the rest against col 0
        for j in range(i):
            M[:, i] -= np.vdot(M[:, j], M[:, i]) * M[:, j]
        nrm = np.linalg.norm(M[:, i])
        if nrm < 1e-9: M[:, i] = np.eye(8)[:, (i+3) % 8]  # replace degenerate col
        M[:, i] /= np.linalg.norm(M[:, i])
    return UnitaryGate(M, label="prep7")

# order-7 QFT embedded in 8 dimensions (acts as identity on the unused |7>)
def qft7():
    w = np.exp(2j*np.pi/7)
    D = np.zeros((8, 8), dtype=complex)
    for i in range(7):
        for j in range(7): D[i, j] = w**(i*j)/np.sqrt(7)
    D[7, 7] = 1
    return UnitaryGate(D, label="QFT7")

print("oracle / prep / QFT helpers ready")

## 3. Build and draw the circuit

In [ ]:
a_reg = QuantumRegister(3, 'a')     # index register a
b_reg = QuantumRegister(3, 'b')     # index register b
pt    = QuantumRegister(6, 'pt')    # point register (encodes aP + bQ)
ca = ClassicalRegister(3, 'ma'); cb = ClassicalRegister(3, 'mb')
qc = QuantumCircuit(a_reg, b_reg, pt, ca, cb)

# point register starts at |O> = code 63 = |111111>
for q in pt: qc.x(q)

# uniform superposition over Z_7 in both index registers
qc.append(prep7(), list(a_reg))
qc.append(prep7(), list(b_reg))
qc.barrier()

# f(a,b) = aP + bQ  via bit-weighted controlled point additions
for j in range(3):
    qc.append(UnitaryGate(controlled_add(ec_mul(2**j, P)), label=f"+{2**j}P"),
              [a_reg[j]] + list(pt))
for j in range(3):
    qc.append(UnitaryGate(controlled_add(ec_mul((2**j) * k_secret, P)), label=f"+{2**j}Q"),
              [b_reg[j]] + list(pt))
qc.barrier()

# QFT_7 on both index registers, then measure
qc.append(qft7(), list(a_reg)); qc.append(qft7(), list(b_reg))
qc.measure(a_reg, ca); qc.measure(b_reg, cb)

qc.draw('mpl', fold=-1)

## 4. Run on Aer and recover the secret $k$

We feed the circuit straight to Aer (its statevector method executes the unitary blocks
directly). Every non-trivial outcome should satisfy $d\equiv kc \pmod 7$ and yield $k=5$.

In [ ]:
sim = AerSimulator()
counts = sim.run(qc, shots=8000).result().get_counts()

# qiskit returns 'mb ma'; convert to integer pairs (c,d)
pairs = Counter()
for bs, n in counts.items():
    mb, ma = bs.split()
    pairs[(int(ma, 2), int(mb, 2))] += n

tot = sum(pairs.values())
recovered = []
print("measured (c, d)   prob    d == k*c (mod 7)?   k = d*c^-1 mod 7")
for (c, d), n in sorted(pairs.items(), key=lambda kv: -kv[1]):
    ok = (d % 7) == (k_secret * c) % 7
    line = f"   ({c}, {d})       {n/tot:5.3f}        {str(ok):<5}"
    if c != 0:
        kk = (d * inv(c, 7)) % 7
        recovered.append((kk, n)); line += f"         {kk}"
    else:
        line += "         (no info)"
    print(line)

k_rec = Counter()
for kk, n in recovered: k_rec[kk] += n
print(f"\nRECOVERED  k = {k_rec.most_common(1)[0][0]}      (true secret k = {k_secret})")

In [ ]:
plot_histogram({f"({c},{d})": n for (c, d), n in pairs.items()},
               figsize=(9, 4),
               title="Shor DLP outcomes (c,d): all satisfy d = 5c mod 7")

## 5. Scaling reality check

The toy needed ~12 qubits for $r=7$. The cost is dominated by two things that both explode
with the field size:

- **Index registers**: $\sim 2\lceil\log_2 r\rceil$ qubits. For 256-bit, $\sim$512 just here.
- **Point register + reversible EC point addition**: representing $(x,y)$ over a 256-bit field
  and doing modular inversion/multiplication *reversibly* is the real engine, and is what pushes
  the total to ~2,330 logical qubits and ~$10^{11}$ Toffoli gates.

In this notebook the point-addition oracle was a precomputed permutation over 7 points. At
256-bit scale you cannot precompute or store that permutation ($2^{256}$ points), and you cannot
hold the statevector ($2^{2330}$ amplitudes). The algorithm is identical; only the *resources*
differ, by hundreds of orders of magnitude. That gap is precisely the security margin of ECC
against quantum attack today.

In [ ]:
import math
for bits, name in [(3, "toy  r=7"), (256, "secp256k1 / P-256")]:
    rbits = bits
    idx_qubits = 2 * math.ceil(math.log2(7 if bits == 3 else 2**256))
    logical = 12 if bits == 3 else 2330
    print(f"{name:20s}: ~{logical} logical qubits, "
          f"statevector dim 2^{logical} "
          f"({'simulable on Aer' if logical < 30 else 'IMPOSSIBLE to simulate'})")

### Takeaway

The quantum routine recovered the secret scalar $k$ from $(P, Q)$ alone — a genuine ECDLP
solved by Shor's algorithm. It scales, *in principle*, to `secp256k1`; it does **not** scale
in practice on any simulator or any quantum computer that exists today. Treat this as a working
illustration of the mechanism, useful for teaching and for testing post-processing logic, not as
an attack on real keys.